In [4]:
import torch
# 创建源张量
src = torch.tensor([[1, 2, 3, 4, 5], [6, 7, 8, 9, 10]])
# 创建索引张量
index = torch.tensor([[0, 1, 2, 0, 1], [0, 1, 1, 2, 2]])
# 创建目标张量
self = torch.zeros(3, 5, dtype=src.dtype)
# 使用scatter_add函数
self.scatter_add_(-1, index, src)
# 输出结果
print(self)

tensor([[ 5,  7,  3,  0,  0],
        [ 6, 15, 19,  0,  0],
        [ 0,  0,  0,  0,  0]])


In [ ]:
import numpy as np

# Create a sample array
x = np.arange(-2, 3)

# Use numpy.flatnonzero to find indices of non-zero elements
non_zero_indices = np.flatnonzero(x)

# Output the indices
print("x: ",x)
print("Indices of non-zero elements:", non_zero_indices)

Indices of non-zero elements: [0 1 3 4]


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy

def get_lcc_size(graph):
    """Get the size of the largest connected component"""
    if graph.vcount() == 0:
        return 0
    components = graph.connected_components()
    return max(components.sizes())


def preprocess(graph, K=10, centrality=None, max_iterations=None, partition_method=None, **kwargs):
    """
    Multi-layer network dismantling baseline
    
    Strategy:
    1. Find the largest connected component (LCC)
    2. Apply community detection on the LCC
    3. Select top node from each community using specified centrality measure
    4. Remove these nodes from the ENTIRE graph (not just LCC)
    
    Args:
        graph: igraph Graph object
        K: number of communities to cluster into
        centrality: centrality function to use (default: Degree)
        max_iterations: maximum number of iterations (layers)
        partition_method: community detection method (default: FastGreedy)
        **kwargs: additional arguments to pass to centrality function
    
    Returns:
        removed_nodes: list of removed node 
        remove_sizes: removed node number sum after each iteration
        lcc_sizes: list of LCC sizes at each iteration
    """
    if centrality is None:
        centrality = Degree
    
    if partition_method is None:
        partition_method = FastGreedy
    
    g = deepcopy(graph)
    # Add static_id attribute to track original node IDs
    g.vs['static_id'] = list(range(g.vcount()))
    
    removed_sizes = [0]
    removed_nodes = []
    lcc_sizes = []
    
    iteration = 0
    if max_iterations is None:
        max_iterations = g.vcount()
    
    while g.vcount() > 0 and g.ecount() > 0 and iteration < max_iterations:
        # Step 1: Get the largest connected component (LCC)
        components = g.connected_components()
        lcc_indices = max(components, key=len)  # Node indices in LCC
        lcc_sizes.append(len(lcc_indices))
        
        # If LCC is too small, stop
        if len(lcc_indices) == 0:
            break
        
        # Step 2: Extract LCC as a subgraph
        lcc_subgraph = g.subgraph(lcc_indices)
        
        # Step 3: Apply community_fastgreedy on the LCC
        # Use the partition function
        membership = partition(lcc_subgraph, K, partition_method=partition_method)
        
        # Step 4: Find the most important node in each community using degree heuristic
        step_nodes = []
        unique_communities = set(membership)
        
        for comm_id in unique_communities:
            comm_nodes = [i for i, m in enumerate(membership) if m == comm_id]
            
            if comm_nodes:
                # Create a subgraph of just this community to apply centrality
                comm_subgraph = lcc_subgraph.subgraph(comm_nodes)
                
                # Use the provided centrality method to find best node in this community
                best_node_in_comm = centrality(comm_subgraph, k=1, **kwargs)[0]
                
                if best_node_in_comm:
                    # Map from community subgraph index -> LCC subgraph index
                    step_id = lcc_indices[comm_nodes[best_node]]
                    step_nodes.append(step_id)

                    # Map from community subgraph index -> original graph index, Use static_id to get the true original ID
                    if 'static_id' in comm_subgraph.vs.attributes():
                        original_id = comm_subgraph.vs[best_node_in_comm]['static_id']
                    else:
                        original_id = lcc_indices[comm_nodes[best_node]]
                    removed_nodes.append(original_id)    
                    
                    assert original_id == g.vs[step_id]['static_id']
        
        # Step 5: Remove the K best nodes from the ENTIRE graph
        if step_nodes:
            g.delete_vertices(step_nodes)
            removed_sizes.append(removed_sizes[-1]+len(nodes_to_remove))
        else:
            break
        
        iteration += 1
    
    lcc_sizes.append(get_lcc_size(g))
    
    return g, removed_nodes, removed_sizes, lcc_sizes

print("Testing process methods...")
g_test = ig.Graph([(0,1),(0,2),(0,3),(1,3),(2,4),(3,4)])
preprocess(g_test, K=2, max_iterations=2)